In [1]:
%pip install -q \
langchain==0.2.17 \
langchain-community==0.2.17 \
langchain-core==0.2.43 \
langchain-text-splitters==0.2.4 \
langsmith==0.1.147 \
langchain-mistralai==0.1.8 \
chromadb==0.5.3 \
sentence-transformers \
pypdf \
gradio==3.50.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.5/559.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.2/299.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.8 MB/

In [2]:
!pip uninstall -y numpy
!pip install numpy==1.26.4

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
yfinance 0.2.66 requires websockets>=13.0, but you have websockets 11.0.3 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ja

In [2]:
%pip install -q \
langchain==0.2.17 \
langchain-community==0.2.17 \
langchain-core==0.2.43 \
langchain-text-splitters==0.2.4 \
langsmith==0.1.147 \
langchain-mistralai==0.1.8 \
chromadb==0.5.3 \
sentence-transformers \
pypdf \
gradio==3.50.2 \
--no-deps

In [9]:
import os
os.environ["MISTRAL_API_KEY"] = "API_KEY_HERE"

In [20]:
from langchain_mistralai import ChatMistralAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.chains import RetrievalQA
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate
import gradio as gr
import os

# Set your API key
os.environ["MISTRAL_API_KEY"] = "API_KEY_HERE"

# -------- Prompt --------
prompt_template = """
You are an AI assistant. Answer ONLY using the provided context.

Context:
{context}

Question:
{question}

Instructions:
- Give concise answer
- Avoid repetition
- Use bullet points if needed
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# -------- LLM --------
def get_llm():
    return ChatMistralAI(
        model="mistral-small",
        temperature=0.2
    )

# -------- Loader --------
def document_loader(file):
    loader = PyPDFLoader(file.name)
    docs = loader.load()

    if len(docs) == 0:
        raise ValueError("No content loaded from PDF")

    return docs

# -------- Split --------
def text_splitter(data):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    chunks = splitter.split_documents(data)

    if len(chunks) == 0:
        raise ValueError("No chunks created")

    return chunks

# -------- Embeddings --------
def get_embeddings():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

# -------- Vector DB --------
def vector_database(chunks):
    return Chroma.from_documents(chunks, get_embeddings())

# -------- Retriever --------
def retriever(file):
    docs = document_loader(file)
    print("Loaded docs:", len(docs))

    chunks = text_splitter(docs)
    print("Chunks:", len(chunks))

    vectordb = vector_database(chunks)

    return vectordb.as_retriever(search_kwargs={"k": 3})

# -------- QA --------
def retriever_qa(file, query):
    try:
        if file is None:
            return "Please upload a PDF first."

        if not query:
            return "Please enter a question."

        llm = get_llm()
        retriever_obj = retriever(file)

        qa = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=retriever_obj,
            return_source_documents=True,
            chain_type_kwargs={"prompt": PROMPT}
        )

        response = qa.invoke({"query": query})

        answer = response["result"]
        sources = response["source_documents"]

        source_text = "\n".join(
            [doc.metadata.get("source", "Unknown") for doc in sources]
        )

        return f"{answer}\n\n---\nSources:\n{source_text}"

    except Exception as e:
        return f"Error: {str(e)}"

# -------- UI --------
demo = gr.Interface(
    fn=retriever_qa,
    inputs=[
        gr.File(label="Upload PDF"),
        gr.Textbox(label="Ask Question")
    ],
    outputs="text",
    title="QA Bot with RAG (Mistral)",
    description="Upload PDF and ask questions"
)

# -------- Launch --------
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://8bee6939ebe62cf11a.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
